# Stage 1 — Data + Signals (Hour 0-2)

Pull OHLCV + Polygon news + Fama-French. Build 8 candidate signals. Tag regime dates.

**This notebook prints diagnostic checks at each step.** The final cell tells you which downstream stages are unblocked. If anything critical is `FAIL`, fix before proceeding.

**Run order:** top to bottom, once. Outputs land in `workspace/data/`.

**Time budget:** 60-90 min, mostly waiting on Polygon (rate-limited free tier).

In [1]:
import os, sys, time, warnings, json
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import requests
import yfinance as yf
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# ---- CONFIG ----
WORKSPACE = '/Applications/Projects/Hackathons/quantum_ai/workspace'
DATA_DIR = os.path.join(WORKSPACE, 'data')
os.makedirs(DATA_DIR, exist_ok=True)

START_DATE = '2023-06-01'
TRAIN_END  = '2025-11-30'
TEST_START = '2025-12-01'

FAST_MODE = False               # Set True for dev: shorter window, skip Polygon
POLYGON_LOOKBACK_DAYS = 90      # free-tier friendly

if FAST_MODE:
    START_DATE = '2025-01-01'
    POLYGON_LOOKBACK_DAYS = 0

# ---- DIAGNOSTIC HELPER ----
CHECKS = {}

def check(name, condition, msg='', critical=True):
    status = 'PASS' if condition else ('FAIL' if critical else 'WARN')
    CHECKS[name] = (status, msg)
    icon = '[PASS]' if status == 'PASS' else ('[FAIL]' if status == 'FAIL' else '[WARN]')
    print(f'{icon} {name}: {msg}')
    return condition

print(f'Workspace: {WORKSPACE}')
print(f'Data dir:  {DATA_DIR}')
print(f'Date range: {START_DATE} → {TRAIN_END} (test starts {TEST_START})')
print(f'Fast mode: {FAST_MODE}')

Workspace: /Applications/Projects/Hackathons/quantum_ai/workspace
Data dir:  /Applications/Projects/Hackathons/quantum_ai/workspace/data
Date range: 2023-06-01 → 2025-11-30 (test starts 2025-12-01)
Fast mode: False


## 0. Universe + sector taxonomy

In [2]:
TICKER_SECTOR = {
    'AAPL': 'tech', 'MSFT': 'tech', 'GOOGL': 'tech', 'NVDA': 'tech',
    'JPM':  'fin',  'BAC':  'fin',  'V':     'fin',  'MA':   'fin',
    'JNJ':  'hc',   'PFE':  'hc',   'UNH':   'hc',   'LLY':  'hc',
    'WMT':  'cons', 'PG':   'cons', 'KO':    'cons', 'PEP':  'cons',
    'XOM':  'eng',  'CVX':  'eng',  'CAT':   'eng',  'BA':   'eng',
}
TICKERS = list(TICKER_SECTOR)

print(f'{len(TICKERS)} tickers across {len(set(TICKER_SECTOR.values()))} sectors:')
for s in sorted(set(TICKER_SECTOR.values())):
    members = [t for t, sec in TICKER_SECTOR.items() if sec == s]
    print(f'  {s:6s}: {members}')

# Save
pd.Series(TICKER_SECTOR, name='sector').to_csv(os.path.join(DATA_DIR, 'ticker_sector.csv'))
check('Universe: 20 tickers, 5 sectors x 4', len(TICKERS) == 20 and len(set(TICKER_SECTOR.values())) == 5,
      f'{len(TICKERS)} tickers, {len(set(TICKER_SECTOR.values()))} sectors')

20 tickers across 5 sectors:
  cons  : ['WMT', 'PG', 'KO', 'PEP']
  eng   : ['XOM', 'CVX', 'CAT', 'BA']
  fin   : ['JPM', 'BAC', 'V', 'MA']
  hc    : ['JNJ', 'PFE', 'UNH', 'LLY']
  tech  : ['AAPL', 'MSFT', 'GOOGL', 'NVDA']
[PASS] Universe: 20 tickers, 5 sectors x 4: 20 tickers, 5 sectors


True

## 1. OHLCV from yfinance

In [3]:
print(f'Pulling OHLCV: {len(TICKERS)} tickers from {START_DATE}...')
raw = yf.download(TICKERS, start=START_DATE, progress=False, auto_adjust=True)

# yfinance returns multi-index columns when >1 ticker. Defensive handling.
if isinstance(raw.columns, pd.MultiIndex):
    prices  = raw['Close']
    highs   = raw['High']
    lows    = raw['Low']
    volumes = raw['Volume']
else:
    raise RuntimeError('Expected multi-index columns from yfinance; got single index')

# Reorder to match TICKERS order, drop tickers that returned nothing
prices  = prices.reindex(columns=TICKERS)
highs   = highs.reindex(columns=TICKERS)
lows    = lows.reindex(columns=TICKERS)
volumes = volumes.reindex(columns=TICKERS)

prices.to_parquet(os.path.join(DATA_DIR, 'prices.parquet'))
highs.to_parquet(os.path.join(DATA_DIR, 'highs.parquet'))
lows.to_parquet(os.path.join(DATA_DIR, 'lows.parquet'))
volumes.to_parquet(os.path.join(DATA_DIR, 'volumes.parquet'))

n_dates = len(prices)
n_missing = prices.isna().sum().sum()
missing_pct = n_missing / prices.size * 100
missing_per_ticker = prices.isna().sum()
tickers_with_data = (missing_per_ticker < len(prices) * 0.5).sum()

print(f'\nDate range: {prices.index[0].date()} -> {prices.index[-1].date()}')
print(f'Trading days: {n_dates}')
print(f'Missing values: {n_missing} ({missing_pct:.1f}%)')
print(f'\nTickers with usable history: {tickers_with_data}/{len(TICKERS)}')
if missing_per_ticker.max() > 5:
    print('Worst tickers:')
    print(missing_per_ticker.sort_values(ascending=False).head(5))

check('OHLCV: enough history (>=500 days)', n_dates >= 500,
      f'{n_dates} trading days')
check('OHLCV: low missing rate (<1%)', missing_pct < 1.0,
      f'{missing_pct:.2f}% missing')
check('OHLCV: all 20 tickers usable', tickers_with_data == 20,
      f'{tickers_with_data}/20 tickers')

Pulling OHLCV: 20 tickers from 2023-06-01...

Date range: 2023-06-01 -> 2026-05-29
Trading days: 751
Missing values: 0 (0.0%)

Tickers with usable history: 20/20
[PASS] OHLCV: enough history (>=500 days): 751 trading days
[PASS] OHLCV: low missing rate (<1%): 0.00% missing
[PASS] OHLCV: all 20 tickers usable: 20/20 tickers


np.True_

In [4]:
returns = prices.pct_change()
returns.to_parquet(os.path.join(DATA_DIR, 'returns.parquet'))

daily_std = returns.std()
ann_vol = daily_std * np.sqrt(252) * 100
ann_ret = returns.mean() * 252 * 100

print('Per-ticker annualized stats:')
stats = pd.DataFrame({'ann_return_%': ann_ret.round(1), 'ann_vol_%': ann_vol.round(1)})
print(stats.to_string())

extreme_vol = ((daily_std > 0.08) | (daily_std < 0.005)).sum()
check('Returns: vols in reasonable range', extreme_vol == 0,
      f'{extreme_vol} tickers with anomalous daily vol')

Per-ticker annualized stats:
        ann_return_%  ann_vol_%
Ticker                         
AAPL            22.2       25.8
MSFT            13.8       23.8
GOOGL           42.4       29.6
NVDA            67.0       46.7
JPM             31.0       22.7
BAC             26.2       24.8
V               15.0       19.7
MA              12.2       19.8
JNJ             17.1       17.3
PFE             -3.4       24.2
UNH              0.3       36.9
LLY             38.3       35.7
WMT             32.3       21.7
PG               4.0       17.2
KO              13.4       15.4
PEP             -2.8       19.4
XOM             17.5       23.0
CVX             12.9       22.6
CAT             54.4       30.9
BA               9.3       34.0
[PASS] Returns: vols in reasonable range: 0 tickers with anomalous daily vol


np.True_

## 2. News + sentiment (Kaggle → DuckDuckGo → Polygon)

Three independent sources. Cell merges them — each fills cells the previous didn't cover.

1. **Kaggle** (`elsabetyemane/financial-news-and-stock-price-integration-dataset`) — multi-year pre-labeled sentiment. Best for training data. Needs `~/.kaggle/kaggle.json`.
2. **DuckDuckGo News** — no auth required, always works. Headline-based sentiment computed locally via TextBlob. Limited to ~recent weeks but no rate limits.
3. **Polygon** — recent ~90 days, pre-tagged sentiment. Optional supplement.

If all three fail → zeros. KTA correctly down-ranks `news_sent` and pipeline still runs.

**If you saw the kagglesdk ImportError last run:** RESTART KERNEL (Kernel → Restart) before re-running this cell. The package was reinstalled but Python cached the broken version.

In [ ]:
import subprocess

# All three sources run independently; combined at end via mean of available values.
source_dfs = {}

def empty_df():
    return pd.DataFrame(0.0, index=prices.index, columns=TICKERS)

# ============================================================
# TIER 1: Kaggle
# ============================================================
print('[Kaggle] loading financial-news dataset...')
try:
    needs_reinstall = False
    try:
        from kagglesdk.kaggle_env import get_web_endpoint  # noqa
        import kagglehub
        from kagglehub import KaggleDatasetAdapter
    except (ImportError, AttributeError) as e:
        needs_reinstall = True
        print(f'[Kaggle] kagglesdk broken ({e})')
        print('[Kaggle] doing aggressive reinstall...')

    if needs_reinstall:
        for pkg in ['kaggle', 'kagglehub', 'kagglesdk']:
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', pkg], check=False)
        subprocess.run([sys.executable, '-m', 'pip', 'cache', 'purge'], check=False)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
                               'kagglesdk>=0.1.10', 'kagglehub[pandas-datasets]>=0.3.0'])
        raise RuntimeError('kagglehub reinstalled with version pin. RESTART KERNEL and re-run.')

    kdf = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        'elsabetyemane/financial-news-and-stock-price-integration-dataset',
        '',
    )
    print(f'[Kaggle] loaded shape={kdf.shape}, columns={list(kdf.columns)}')

    def find_col(df, candidates):
        for cand in candidates:
            for col in df.columns:
                if cand.lower() in str(col).lower():
                    return col
        return None

    date_col      = find_col(kdf, ['date', 'timestamp', 'published'])
    ticker_col    = find_col(kdf, ['ticker', 'symbol', 'stock'])
    sentiment_col = find_col(kdf, ['sentiment_score', 'compound', 'sentiment_label', 'sentiment', 'label', 'polarity'])
    print(f'[Kaggle] detected: date={date_col}, ticker={ticker_col}, sentiment={sentiment_col}')

    if not (date_col and ticker_col and sentiment_col):
        raise RuntimeError(f'column auto-detection failed (cols={list(kdf.columns)})')

    kdf[date_col] = pd.to_datetime(kdf[date_col], errors='coerce')
    kdf = kdf.dropna(subset=[date_col])
    if kdf[date_col].dt.tz is not None:
        kdf[date_col] = kdf[date_col].dt.tz_localize(None)
    kdf[date_col] = kdf[date_col].dt.normalize()

    if kdf[sentiment_col].dtype == object:
        smap = {'positive': 1, 'negative': -1, 'neutral': 0,
                'POSITIVE': 1, 'NEGATIVE': -1, 'NEUTRAL': 0,
                'Positive': 1, 'Negative': -1, 'Neutral': 0,
                'pos': 1, 'neg': -1, 'neu': 0}
        kdf['_sent_num'] = kdf[sentiment_col].map(smap).fillna(0)
    else:
        kdf['_sent_num'] = pd.to_numeric(kdf[sentiment_col], errors='coerce').fillna(0).clip(-1, 1)

    kdf['_ticker_norm'] = kdf[ticker_col].astype(str).str.upper().str.strip()
    kdf_sub = kdf[kdf['_ticker_norm'].isin(TICKERS)]
    print(f'[Kaggle] rows for our 20 tickers: {len(kdf_sub)} (of {len(kdf)} total)')

    if len(kdf_sub) > 0:
        agg = (kdf_sub.groupby([date_col, '_ticker_norm'])['_sent_num']
                      .mean().unstack(fill_value=0))
        agg = agg.reindex(index=prices.index, columns=TICKERS, fill_value=0)
        source_dfs['kaggle'] = agg.copy()
        print(f'[Kaggle] coverage: {int((agg != 0).sum().sum())} cells')

except Exception as e:
    print(f'[Kaggle] FAILED: {type(e).__name__}: {e}')

# ============================================================
# TIER 2: DDG (uses NEW `ddgs` package, not legacy duckduckgo_search)
# ============================================================
print('\n[DDG] pulling headlines for sentiment...')
try:
    try:
        from ddgs import DDGS
    except ImportError:
        print('[DDG] installing ddgs (new package, replaces duckduckgo_search)...')
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'duckduckgo_search'], check=False)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ddgs'])
        from ddgs import DDGS

    try:
        from textblob import TextBlob
    except ImportError:
        print('[DDG] installing textblob...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'textblob'])
        from textblob import TextBlob

    ddg_sent = empty_df()
    ddg_errors = 0

    for i, ticker in enumerate(TICKERS):
        try:
            with DDGS() as ddg_client:
                results = list(ddg_client.news(f'{ticker} stock', max_results=30, region='us-en'))
            for r in results:
                try:
                    d_pub = pd.to_datetime(r.get('date'))
                    if d_pub is None or pd.isna(d_pub):
                        continue
                    if d_pub.tz is not None:
                        d_pub = d_pub.tz_localize(None)
                    d = d_pub.normalize()
                    if d not in ddg_sent.index:
                        continue
                    text = (r.get('title') or '') + '. ' + (r.get('body') or '')
                    if not text.strip():
                        continue
                    polarity = float(TextBlob(text).sentiment.polarity)
                    ddg_sent.loc[d, ticker] += polarity
                except Exception:
                    continue
            time.sleep(1.5)
        except Exception:
            ddg_errors += 1
            time.sleep(3)
            continue
        if (i + 1) % 5 == 0:
            print(f'  ...{i + 1}/{len(TICKERS)}')

    ddg_sent = ddg_sent.clip(-1, 1)
    n_ddg = int((ddg_sent != 0).sum().sum())
    print(f'[DDG] coverage: {n_ddg} cells, errors: {ddg_errors}')

    if n_ddg > 0:
        source_dfs['ddg'] = ddg_sent

except Exception as e:
    print(f'[DDG] FAILED: {type(e).__name__}: {e}')

# ============================================================
# TIER 3: Polygon (optional)
# ============================================================
POLYGON_KEY = os.environ.get('POLYGON_API_KEY')

if POLYGON_KEY and POLYGON_LOOKBACK_DAYS > 0:
    cutoff = (datetime.now() - timedelta(days=POLYGON_LOOKBACK_DAYS)).strftime('%Y-%m-%d')
    print(f'\n[Polygon] supplementing from {cutoff}...')

    polygon_sent = empty_df()
    polygon_errors = 0

    for i, ticker in enumerate(TICKERS):
        url = 'https://api.polygon.io/v2/reference/news'
        params = {'ticker': ticker, 'limit': 50,
                  'published_utc.gte': cutoff, 'apiKey': POLYGON_KEY}
        try:
            r = requests.get(url, params=params, timeout=20)
            if r.status_code == 429:
                time.sleep(20)
                r = requests.get(url, params=params, timeout=20)
            if r.status_code != 200:
                polygon_errors += 1
                time.sleep(13)
                continue
            for art in r.json().get('results', []):
                d_pub = pd.to_datetime(art.get('published_utc'))
                d = d_pub.tz_localize(None).normalize() if d_pub.tz is not None else d_pub.normalize()
                if d not in polygon_sent.index:
                    continue
                for ins in art.get('insights', []) or []:
                    if ins.get('ticker') == ticker:
                        sc = {'positive': 1, 'neutral': 0, 'negative': -1}.get(
                            ins.get('sentiment', 'neutral'), 0)
                        polygon_sent.loc[d, ticker] += sc
            time.sleep(13)
        except Exception:
            polygon_errors += 1
            time.sleep(13)
        if (i + 1) % 5 == 0:
            print(f'  ...{i + 1}/{len(TICKERS)}')

    polygon_sent = polygon_sent.clip(-1, 1)
    n_pg = int((polygon_sent != 0).sum().sum())
    print(f'[Polygon] coverage: {n_pg} cells, errors: {polygon_errors}')

    if n_pg > 0:
        source_dfs['polygon'] = polygon_sent
else:
    print('\n[Polygon] skipped (no key)')

# ============================================================
# COMBINE — mean of non-zero values per cell across sources
# ============================================================
print('\n' + '=' * 60)
print('COMBINING SOURCES')
print('=' * 60)

if source_dfs:
    sum_df = sum(df for df in source_dfs.values())
    count_df = sum((df != 0).astype(int) for df in source_dfs.values())
    news_sent = (sum_df / count_df.replace(0, 1)).where(count_df > 0, 0)
else:
    news_sent = empty_df()

news_sent.to_parquet(os.path.join(DATA_DIR, 'news_sent.parquet'))

print('\nPer-source coverage:')
print(f'{"source":12s} {"cells":>10s}  {"% grid":>8s}  {"mean |sent|":>12s}')
print('-' * 50)
for src, df in source_dfs.items():
    n = int((df != 0).sum().sum())
    pct = n / df.size * 100
    nz = df.values[df.values != 0]
    mean_abs = float(np.nanmean(np.abs(nz))) if len(nz) > 0 else 0
    print(f'{src:12s} {n:>10d}  {pct:>7.2f}%  {mean_abs:>12.4f}')

if len(source_dfs) >= 2:
    print('\nSource overlap (cells with multiple contributions):')
    counts = sum((df != 0).astype(int) for df in source_dfs.values())
    for k in range(1, len(source_dfs) + 1):
        n_k = int((counts == k).sum().sum())
        print(f'  {k} source(s):  {n_k} cells')

n_combined = int((news_sent != 0).sum().sum())
sources_str = ' + '.join(sorted(source_dfs.keys())) if source_dfs else 'NONE'
print(f'\nFINAL combined news_sent:')
print(f'  Sources: {sources_str}')
print(f'  Total non-zero cells: {n_combined} ({n_combined / news_sent.size * 100:.2f}% of grid)')

check(f'News: combined source={sources_str}', n_combined > 0,
      f'{n_combined} cells from {len(source_dfs)} source(s)', critical=False)

## 3. Fama-French 5-factor (free, Tuck)

In [ ]:
print('Pulling Fama-French 5-factor daily...')
ff = None
try:
    try:
        from pandas_datareader.famafrench import FamaFrenchReader
    except ImportError:
        print('Installing pandas-datareader...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pandas-datareader'])
        from pandas_datareader.famafrench import FamaFrenchReader

    reader = FamaFrenchReader('F-F_Research_Data_5_Factors_2x3_daily',
                              start=START_DATE)
    ff = reader.read()[0] / 100.0
    ff.index = pd.to_datetime(ff.index)
    ff = ff.reindex(prices.index).ffill()
    print(f'Loaded FF factors: {list(ff.columns)}')
    print(f'Rows: {len(ff)}, NaN rows: {ff.isna().any(axis=1).sum()}')
    check('Fama-French: loaded', True, f'{ff.shape[1]} factors, {len(ff)} rows')
except Exception as e:
    print(f'Fama-French pull failed: {type(e).__name__}: {e}')
    print('Falling back to zero factors (factor signals will be weak)')
    ff = pd.DataFrame(0.0, index=prices.index,
                       columns=['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF'])
    check('Fama-French: loaded', False, f'fallback: {e}', critical=False)

ff.to_parquet(os.path.join(DATA_DIR, 'fama_french.parquet'))

## 4. Build the 8 candidate signals

In [8]:
def rsi(series, n=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0).rolling(n).mean()
    loss = -delta.where(delta < 0, 0).rolling(n).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

SIGNALS = ['mom_5d', 'mom_20d', 'rev_1d', 'vol_20d', 'rsi_14',
           'news_sent', 'vol_ratio', 'spread_prox']

signal_panels = {}
for t in TICKERS:
    r = returns[t]
    p = prices[t]
    sig = pd.DataFrame(index=prices.index)
    sig['mom_5d']       = r.rolling(5).sum()
    sig['mom_20d']      = r.rolling(20).sum()
    sig['rev_1d']       = -r
    sig['vol_20d']      = r.rolling(20).std()
    sig['rsi_14']       = rsi(p)
    sig['news_sent']    = news_sent[t].rolling(3).mean()
    sig['vol_ratio']    = r.rolling(5).std() / r.rolling(60).std().replace(0, np.nan)
    sig['spread_prox']  = (highs[t] - lows[t]) / p
    signal_panels[t] = sig

# Stack into multi-index DataFrame: columns=(ticker, signal)
signals_long = pd.concat({t: signal_panels[t] for t in TICKERS}, axis=1)
signals_long.to_parquet(os.path.join(DATA_DIR, 'signals.parquet'))

print(f'Signals shape: {signals_long.shape}')
print(f'Columns sample: {list(signals_long.columns[:5])}')
print(f'Date range: {signals_long.index[0].date()} -> {signals_long.index[-1].date()}')

check('Signals: 8 per ticker built', signals_long.shape[1] == len(TICKERS) * len(SIGNALS),
      f'{signals_long.shape[1]} columns ({len(TICKERS)} x {len(SIGNALS)})')

Signals shape: (751, 160)
Columns sample: [('AAPL', 'mom_5d'), ('AAPL', 'mom_20d'), ('AAPL', 'rev_1d'), ('AAPL', 'vol_20d'), ('AAPL', 'rsi_14')]
Date range: 2023-06-01 -> 2026-05-29
[PASS] Signals: 8 per ticker built: 160 columns (20 x 8)


True

In [ ]:
# Per-signal NaN rate + variance — is each signal usable?
print('Per-signal usability check:')
print(f'{"signal":12s} {"NaN %":>7s}  {"std":>10s}  {"valid rows":>12s}  {"verdict":>10s}')
print('-' * 65)

for s in SIGNALS:
    cols_for_s = [c for c in signals_long.columns if c[1] == s]
    sub = signals_long.loc[:, cols_for_s]
    nan_pct = float(sub.isna().sum().sum()) / sub.size * 100
    std = float(np.nanstd(sub.values))               # scalar — avoids Series ambiguity
    valid_rows = int(sub.dropna(how='all').shape[0])
    usable = bool((nan_pct < 30) and (std > 1e-6))
    verdict = 'OK' if usable else 'WEAK'
    print(f'{s:12s} {nan_pct:7.1f}  {std:10.4f}  {valid_rows:12d}  {verdict:>10s}')
    check(f'Signal {s}: usable', usable, f'NaN={nan_pct:.1f}%, std={std:.4f}',
          critical=(s != 'news_sent'))   # news_sent failure is non-critical

## 5. Regime tags (for Q-Atlas demo)

Tag known event dates so Q-Atlas can show analogs with human-readable labels. Quick heuristic: abnormal-volume days (likely earnings) + FOMC-like Wednesdays.

In [ ]:
regime_tags = pd.Series('', index=prices.index, name='regime_tag', dtype=object)

# 1. Earnings-like days: vol z-score > 3
for t in TICKERS:
    vol_z = (volumes[t] - volumes[t].rolling(30).mean()) / volumes[t].rolling(30).std()
    spike_days = vol_z[vol_z > 3.0].index
    for d in spike_days:
        existing = regime_tags.loc[d]
        regime_tags.loc[d] = f'{existing}, {t}_volspike'.strip(', ') if existing else f'{t}_volspike'

# 2. FOMC days (3rd Wed of FOMC months — rough heuristic)
fomc_months = [1, 3, 5, 6, 7, 9, 11, 12]
for year in range(int(START_DATE[:4]), 2027):
    for m in fomc_months:
        d = pd.Timestamp(year, m, 1)
        wed_count = 0
        while d.month == m:
            if d.weekday() == 2:
                wed_count += 1
                if wed_count == 3:
                    if d in regime_tags.index:
                        existing = regime_tags.loc[d]
                        regime_tags.loc[d] = f'{existing}, FOMC'.strip(', ') if existing else 'FOMC'
                    break
            d += pd.Timedelta(days=1)

regime_tags.to_frame().to_parquet(os.path.join(DATA_DIR, 'regime_tags.parquet'))

n_tagged = (regime_tags != '').sum()
print(f'Tagged {n_tagged} dates')
print('\nSample tagged dates:')
print(regime_tags[regime_tags != ''].head(15).to_string())

check('Regime tags: >=20 dates labeled', n_tagged >= 20,
      f'{n_tagged} dates have regime labels (more = better Q-Atlas demos)',
      critical=False)

## 6. Forward returns (Stage 2/3 labels)

In [ ]:
fwd_1d = returns.shift(-1)
fwd_5d = returns.rolling(5).sum().shift(-5)

fwd_1d.to_parquet(os.path.join(DATA_DIR, 'forward_returns_1d.parquet'))
fwd_5d.to_parquet(os.path.join(DATA_DIR, 'forward_returns_5d.parquet'))

pos_rate = (fwd_1d > 0).stack().mean()
n_valid_1d = fwd_1d.dropna(how='all').shape[0]
n_valid_5d = fwd_5d.dropna(how='all').shape[0]

print(f'1d forward returns: {n_valid_1d} valid days')
print(f'5d forward returns: {n_valid_5d} valid days')
print(f'1d positive return rate: {pos_rate:.2%}')

check('Forward labels: balanced classes', 0.40 < pos_rate < 0.60,
      f'{pos_rate:.2%} positive (need [40%, 60%] for stable meta-classifier)')

## 7. Signal usefulness — IC + correlation matrix

The critical check. For each of the 8 signals, compute Information Coefficient (Spearman correlation between signal value and next-day return). Then compute pairwise signal-signal correlations to verify diversity.

**Stage 2 needs at least 3 signals with |IC| > 0.01** to have something to learn from.

In [ ]:
ic_results = {}

print(f'{"signal":12s} {"mean IC":>10s}  {"|IC|":>8s}  {"IC stderr":>10s}  {"IR-like":>10s}')
print('-' * 60)

for s in SIGNALS:
    ics = []
    for t in TICKERS:
        sig_vals = signal_panels[t][s]
        fwd_vals = fwd_1d[t]
        df_pair = pd.concat([sig_vals, fwd_vals], axis=1).dropna()
        if len(df_pair) < 30:
            continue
        rho, _ = spearmanr(df_pair.iloc[:, 0], df_pair.iloc[:, 1])
        if not np.isnan(rho):
            ics.append(rho)
    if not ics:
        ic_results[s] = np.nan
        print(f'{s:12s}  no valid data')
        continue
    mean_ic = np.mean(ics)
    se = np.std(ics) / np.sqrt(len(ics))
    ir_like = abs(mean_ic) / (np.std(ics) + 1e-9)
    ic_results[s] = mean_ic
    print(f'{s:12s} {mean_ic:10.4f}  {abs(mean_ic):8.4f}  {se:10.4f}  {ir_like:10.2f}')

n_useful = sum(1 for s, ic in ic_results.items() if abs(ic) > 0.01)
print(f'\nSignals with |IC| > 0.01: {n_useful} / {len(SIGNALS)}')

check('Signals: >=3 with predictive power', n_useful >= 3,
      f'{n_useful}/{len(SIGNALS)} have |IC| > 0.01 (need >=3 for Stage 2 to work)')

# Save IC ranking — KTA in Stage 1.5 will compare against this
ic_series = pd.Series(ic_results).abs().sort_values(ascending=False)
ic_series.to_csv(os.path.join(DATA_DIR, 'ic_ranking.csv'))
print(f'\nTop signals by |IC|:')
print(ic_series.head(6).to_string())

In [ ]:
# Signal-signal correlation matrix (avg across tickers)
print('Pairwise signal correlation (avg across tickers):')
sig_corr = pd.DataFrame(index=SIGNALS, columns=SIGNALS, dtype=float)

for s1 in SIGNALS:
    for s2 in SIGNALS:
        rhos = []
        for t in TICKERS:
            df_pair = pd.concat([signal_panels[t][s1], signal_panels[t][s2]], axis=1).dropna()
            if len(df_pair) < 30:
                continue
            rho, _ = spearmanr(df_pair.iloc[:, 0], df_pair.iloc[:, 1])
            if not np.isnan(rho):
                rhos.append(rho)
        sig_corr.loc[s1, s2] = np.mean(rhos) if rhos else np.nan

print(sig_corr.round(2).to_string())
sig_corr.to_csv(os.path.join(DATA_DIR, 'signal_correlations.csv'))

# Check: top 5-6 signals by IC should not all be highly correlated
top_signals = ic_series.head(6).index.tolist()
top_corr = sig_corr.loc[top_signals, top_signals].abs()
# Off-diagonal max
np.fill_diagonal(top_corr.values, 0)
max_offdiag = top_corr.max().max()

print(f'\nTop-6 signals: {top_signals}')
print(f'Max off-diagonal correlation among top-6: {max_offdiag:.2f}')
check('Signal diversity: top-6 not over-correlated', max_offdiag < 0.85,
      f'max pairwise |corr|={max_offdiag:.2f} (need <0.85 so KTA picks diverse winners)',
      critical=False)

## 8. Stage readiness summary

Final GO/NO-GO per downstream stage. **If a stage shows STOP, fix the failing checks before proceeding.**

In [ ]:
def stage_status(prereq_prefixes):
    relevant = {n: v for n, v in CHECKS.items()
                if any(n.startswith(p) for p in prereq_prefixes)}
    fails = [(n, v[1]) for n, v in relevant.items() if v[0] == 'FAIL']
    warns = [(n, v[1]) for n, v in relevant.items() if v[0] == 'WARN']
    if fails:
        return 'STOP', f'{len(fails)} failed', fails
    if warns:
        return 'GO (with warnings)', f'{len(warns)} warnings', warns
    return 'GO', 'all green', []

print('=' * 72)
print(' STAGE READINESS SUMMARY')
print('=' * 72)

stages = [
    ('Stage 2 (Alpha models)',
     ['OHLCV', 'Returns', 'Signals', 'Signal ', 'Forward'],
     'needs OHLCV + computable signals + forward returns'),
    ('Stage 3 (Meta-labeling) — HEADLINE',
     ['Signals', 'Signal ', 'Forward'],
     'needs signal variety + balanced labels'),
    ('Stage 4 (QAOA portfolio)',
     ['OHLCV', 'Returns', 'Universe'],
     'needs returns for covariance + sector tags'),
    ('Stage 5.5 (Q-Atlas demo)',
     ['Regime'],
     'needs regime-tagged dates for human-readable analogs'),
]

for name, prefixes, desc in stages:
    status, msg, issues = stage_status(prefixes)
    print(f'\n  {name}')
    print(f'    prereqs: {desc}')
    print(f'    --> {status}: {msg}')
    for n, m in issues:
        print(f'         - {n}: {m}')

print()
print('=' * 72)
all_critical_fail = [n for n, (s, _) in CHECKS.items() if s == 'FAIL']
if all_critical_fail:
    print(f' OVERALL: RED — {len(all_critical_fail)} critical check(s) failed')
    print(' Fix these before moving to Stage 2:')
    for n in all_critical_fail:
        print(f'   - {n}: {CHECKS[n][1]}')
else:
    print(' OVERALL: GREEN — data is ready for Stage 2 (alpha models)')
    print(f'   Outputs in {DATA_DIR}:')
    print('     - prices.parquet, returns.parquet, volumes.parquet')
    print('     - news_sent.parquet, fama_french.parquet')
    print('     - signals.parquet, regime_tags.parquet')
    print('     - forward_returns_1d.parquet, forward_returns_5d.parquet')
    print('     - ticker_sector.csv, ic_ranking.csv, signal_correlations.csv')
print('=' * 72)

# Persist the check log for downstream notebooks
with open(os.path.join(DATA_DIR, 'stage1_checks.json'), 'w') as f:
    json.dump({n: {'status': s, 'msg': m} for n, (s, m) in CHECKS.items()}, f, indent=2)